# پروژه مسیریابی شهری با الگوریتم BFS

این نوت‌بوک، چالش «تبدیل محیط به شبکه دوبعدی» را از تمرین عامل جاروبرقی به مسئله مسیریابی شهری توسعه می‌دهد.

**فرض‌ها:** خانه `0` مسیر قابل عبور و خانه `1` مانع است؛ حرکت فقط به بالا، پایین، چپ و راست مجاز است؛ هزینه همه حرکت‌ها برابر است. بنابراین BFS در صورت وجود مسیر، کوتاه‌ترین مسیر از نظر تعداد حرکت را پیدا می‌کند.

در این نسخه، علاوه بر نمایش مسیر، زمان اجرا و تعداد گره‌های گسترش‌یافته ثبت می‌شود و سه سناریو بررسی می‌شوند: مسیر معمولی، وجود مسیر جایگزین، و نبود مسیر.

In [ ]:
from collections import deque
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm


def bfs_route(grid, start, goal):
    """Find a shortest path in a 4-directional grid using BFS.

    grid: 2D array; 0 = free cell, 1 = obstacle
    start, goal: (row, column)
    Returns a dictionary with path and evaluation metrics.
    """
    grid = np.asarray(grid)
    if grid.ndim != 2 or grid.size == 0:
        raise ValueError("Grid must be a non-empty 2D array.")

    rows, cols = grid.shape

    def inside(point):
        r, c = point
        return 0 <= r < rows and 0 <= c < cols

    if not inside(start) or not inside(goal):
        raise ValueError("Start and goal must be inside the grid.")
    if grid[start] == 1 or grid[goal] == 1:
        raise ValueError("Start and goal must be on free cells (value 0).")

    started_at = perf_counter()
    queue = deque([start])
    parent = {start: None}  # also marks discovered cells
    expanded_nodes = 0
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    found = False

    while queue:
        current = queue.popleft()
        expanded_nodes += 1

        if current == goal:
            found = True
            break

        for dr, dc in directions:
            neighbor = (current[0] + dr, current[1] + dc)
            r, c = neighbor
            if (0 <= r < rows and 0 <= c < cols
                    and grid[neighbor] == 0
                    and neighbor not in parent):
                parent[neighbor] = current
                queue.append(neighbor)

    elapsed_seconds = perf_counter() - started_at

    path = None
    if found:
        path = []
        current = goal
        while current is not None:
            path.append(current)
            current = parent[current]
        path.reverse()

    return {
        "path": path,
        "found": found,
        "moves": len(path) - 1 if path is not None else None,
        "expanded_nodes": expanded_nodes,
        "elapsed_seconds": elapsed_seconds,
    }


def is_valid_path(grid, path, start, goal):
    """Check path endpoints, bounds, obstacles, and adjacent moves."""
    if path is None or len(path) == 0:
        return False
    rows, cols = grid.shape
    if path[0] != start or path[-1] != goal:
        return False
    for r, c in path:
        if not (0 <= r < rows and 0 <= c < cols) or grid[r, c] == 1:
            return False
    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        if abs(r1-r2) + abs(c1-c2) != 1:
            return False
    return True


def plot_route(grid, start, goal, result, title):
    """Display free cells, obstacles, route, start, and goal."""
    shown = np.array(grid, dtype=int).copy()
    path = result["path"]
    if path is not None:
        for cell in path:
            if cell != start and cell != goal:
                shown[cell] = 2
    shown[start] = 3
    shown[goal] = 4

    cmap = ListedColormap(["white", "dimgray", "cornflowerblue", "limegreen", "tomato"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(shown, cmap=cmap, norm=norm)
    for r in range(grid.shape[0]):
        for c in range(grid.shape[1]):
            if (r, c) == start:
                label = "S"
            elif (r, c) == goal:
                label = "G"
            elif grid[r, c] == 1:
                label = "#"
            elif path is not None and (r, c) in path:
                label = "*"
            else:
                label = ""
            ax.text(c, r, label, ha="center", va="center", color="black", fontsize=10)
    ax.set_title(title)
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.set_xticks(np.arange(-.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, grid.shape[0], 1), minor=True)
    ax.grid(which="minor", color="lightgray", linewidth=0.7)
    ax.tick_params(which="minor", bottom=False, left=False)
    plt.show()


## سناریوهای آزمایش

- **سناریو ۱ — مسیر معمولی:** نقشه شهری با موانع پراکنده.
- **سناریو ۲ — مسیر جایگزین:** مسیر مستقیم با مانع بسته شده است؛ عامل باید راه دیگری پیدا کند.
- **سناریو ۳ — نبود مسیر:** مقصد با موانع احاطه شده است؛ انتظار داریم الگوریتم با پیام نبود مسیر خاتمه یابد.

In [ ]:
scenarios = {
    "Scenario 1 - Normal city map": {
        "grid": np.array([
            [0, 0, 0, 1, 0, 0, 0, 0],
            [0, 1, 0, 1, 0, 1, 1, 0],
            [0, 1, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 1, 1, 1, 0, 1],
            [1, 1, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 1, 1, 0],
            [0, 1, 0, 0, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 0, 1, 0],
        ]),
        "start": (0, 0), "goal": (7, 7),
    },
    "Scenario 2 - Detour required": {
        "grid": np.array([
            [0, 0, 1, 0, 0],
            [0, 0, 1, 0, 0],
            [0, 0, 0, 0, 0],
            [1, 1, 0, 1, 1],
            [0, 0, 0, 0, 0],
        ]),
        "start": (0, 0), "goal": (0, 4),
    },
    "Scenario 3 - No possible route": {
        "grid": np.array([
            [0, 0, 0, 0, 0],
            [0, 1, 1, 1, 0],
            [0, 1, 0, 1, 0],
            [0, 1, 1, 1, 0],
            [0, 0, 0, 0, 0],
        ]),
        "start": (0, 0), "goal": (2, 2),
    },
}

results = {}
for name, case in scenarios.items():
    result = bfs_route(case["grid"], case["start"], case["goal"])
    results[name] = result
    print(f"\n{name}")
    print("Path found:", result["found"])
    print("Number of moves:", result["moves"])
    print("Expanded nodes:", result["expanded_nodes"])
    print(f"Execution time: {result['elapsed_seconds']:.8f} seconds")
    if result["path"] is not None:
        print("Path:", result["path"])
        print("Path valid:", is_valid_path(case["grid"], result["path"], case["start"], case["goal"]))
    else:
        print("Path: None (no route exists)")
    plot_route(case["grid"], case["start"], case["goal"], result, name)


## تفسیر نتایج

- در سناریوی اول، عامل باید یک مسیر معتبر به مقصد پیدا کند.
- در سناریوی دوم، وجود مانع در مسیر مستقیم باعث می‌شود عامل مسیر جایگزین پیدا کند.
- در سناریوی سوم، مقصد محصور است و عامل باید اعلام کند که مسیری وجود ندارد.
- **تعداد حرکت‌ها** برابر تعداد خانه‌های مسیر منهای یک است.
- **گره‌های گسترش‌یافته** تعداد موقعیت‌هایی است که از صف خارج و بررسی شده‌اند.
- **زمان اجرا** در هر اجرا اندازه‌گیری می‌شود و به سخت‌افزار و شرایط اجرا وابسته است؛ برای گزارش دقیق‌تر، هر سناریو را چند بار اجرا و میانگین زمان را محاسبه کنید.

### محدودیت‌ها

این مدل نقشه‌ای ساده و ایستا دارد؛ هزینه تمام حرکت‌ها برابر است و ترافیک، خیابان یک‌طرفه و داده‌های واقعی GPS در آن مدل نشده‌اند. برای مسیرهایی با هزینه متفاوت، می‌توان در مرحله بعد از UCS یا A* استفاده کرد.